# B2 — Fine-tune StyleTTS2-lite-vi on Ngạn voice (Kaggle)

**Mục đích**: Fine-tune pretrained StyleTTS2-lite-vi trên ~2155 sample giọng Bác Ngạn để clone giọng cho audiobook horror.

**Workflow**:
1. Phiên 1: Train từ pretrained → epoch 0 → 15
2. Phiên 2: Resume → epoch 15 → 30
3. Phiên 3: Resume → epoch 30 → 45 (nếu cần)
4. Mỗi phiên 7-8 giờ → save version → download checkpoint

**Prerequisites** (đã làm trong B1):
- ✅ Kaggle account verified phone
- ✅ Dataset `vieanh88/ngan-data-lite-vi` đã upload (~1.23 GB)
- ✅ Notebook đã attach dataset vào Input
- ✅ Settings: GPU **T4 ×2**, Internet ON, Persistence "Files only"

> ⚠️ **TUYỆT ĐỐI KHÔNG dùng P100** (PyTorch 2.10 đã drop sm_60).  
> ⚠️ Đặt timer điện thoại **7 giờ** sau khi cell training bắt đầu — phải Save Version trước khi Kaggle kill session (~9h).


## 🔖 Phiên hiện tại

**Điền thông tin bạn đang ở phiên nào:**

| Field | Phiên 1 (lần đầu) | Phiên 2+ (resume) |
|-------|--------------------|--------------------|
| `IS_RESUME` | `False` | `True` |
| `PREVIOUS_CKPT_DATASET` | (bỏ qua) | slug dataset chứa checkpoint phiên trước, vd `"vieanh88/ngan-checkpoint-v1"` |
| `PREVIOUS_CKPT_FILENAME` | (bỏ qua) | tên file, vd `"current_model.pth"` |

Sẽ set ở Cell config bên dưới.

## Bước 1 — GPU compatibility check

In [ ]:
!nvidia-smi
import torch, sys, platform

print(f"\nPython  : {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"PyTorch : {torch.__version__}")

if not torch.cuda.is_available():
    raise RuntimeError("❌ KHÔNG có GPU — vào Settings → Accelerator → GPU T4 x2")

n_gpu = torch.cuda.device_count()
print(f"\nSố GPU detected: {n_gpu}")
for i in range(n_gpu):
    cap = torch.cuda.get_device_capability(i)
    name = torch.cuda.get_device_name(i)
    sm = f"sm_{cap[0]}{cap[1]}"
    supported = torch.cuda.get_arch_list()
    vram_gb = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name}  ({sm}, {vram_gb:.1f} GB)")
    if sm not in str(supported):
        raise RuntimeError(
            f"❌ GPU {name} ({sm}) không được PyTorch {torch.__version__} hỗ trợ.\n"
            f"   Supported: {supported}\n"
            f"   FIX: Settings → Accelerator → đổi sang 'GPU T4 x2' → Save Session."
        )

if n_gpu < 2:
    print(f"\n⚠️  CẢNH BÁO: chỉ có {n_gpu} GPU, expected 2 (T4 x2).")
    print("   Notebook vẫn chạy được nhưng DataParallel sẽ không tăng tốc.")
else:
    print(f"\n✅ T4 x2 detected — DataParallel sẽ tự split batch.")


## Bước 2 — Verify input dataset đã attach

Dataset `ngan-data-lite-vi` phải mount tại `/kaggle/input/ngan-data-lite-vi/`. Nếu KHÔNG thấy → sidebar Input → +Add Data → search và attach lại.

In [ ]:
import os
from pathlib import Path

DATA_ROOT = "/kaggle/input/datasets/vieanh88/ngan-data-lite-vi/ngan-data-lite-vi"

if not Path(DATA_ROOT).exists():
    raise RuntimeError(
        f"❌ KHÔNG thấy {DATA_ROOT}\n"
        f"   Sidebar Input → +Add Data → search 'ngan-data-lite-vi' → Add"
    )

# Verify cấu trúc
TRAIN_FILE = f"{DATA_ROOT}/ngan_train_lite.txt"
VAL_FILE = f"{DATA_ROOT}/ngan_val_lite.txt"
WAVS_DIR = f"{DATA_ROOT}/wavs"

for p in [TRAIN_FILE, VAL_FILE, WAVS_DIR]:
    if not Path(p).exists():
        raise RuntimeError(f"❌ Thiếu: {p}")

n_wavs = len(list(Path(WAVS_DIR).glob("*.wav")))
with open(TRAIN_FILE, encoding="utf-8") as f:
    n_train = sum(1 for _ in f)
with open(VAL_FILE, encoding="utf-8") as f:
    n_val = sum(1 for _ in f)

print(f"✅ Dataset OK:")
print(f"  Wav files: {n_wavs:,}")
print(f"  Train lines: {n_train:,}")
print(f"  Val lines:   {n_val:,}")

# In sample 2 dòng để verify format
print(f"\nSample 2 dòng train:")
with open(TRAIN_FILE, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 2: break
        print(f"  {line.strip()[:150]}")


## Bước 3 — Clone source code StyleTTS2-lite

Repo gốc của tác giả `dangtr0408/StyleTTS2-lite` chứa train.py + models.py + Modules/ + meldataset.py.

In [ ]:
import os

WORK_DIR = "/kaggle/working"
REPO_DIR = f"{WORK_DIR}/StyleTTS2-lite"

os.chdir(WORK_DIR)

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/dangtr0408/StyleTTS2-lite.git
else:
    print(f"Repo đã clone, skip: {REPO_DIR}")

# Verify cấu trúc
%cd {REPO_DIR}
!ls -la
print("\nNội dung Modules/:")
!ls Modules/
print("\nNội dung Configs/:")
!ls Configs/ 2>/dev/null || echo "(folder Configs/ chưa có — sẽ tạo)"


## Bước 4 — Cài dependencies

Theo `requirements.txt` của lite-vi + 1 vài cái thiếu (click, tensorboard, vv).

In [ ]:
# Kaggle có sẵn: torch, torchaudio, numpy, librosa, PyYAML, nltk, tensorboard
# Cần thêm:
!pip install -q munch noisereduce phonemizer espeakng-loader click 2>&1 | tail -3
!apt-get install -y -qq espeak-ng espeak-ng-data 2>&1 | tail -1

# Verify
from importlib.metadata import version, PackageNotFoundError

import munch
import noisereduce
import phonemizer
import click

def get_package_version(package_name: str) -> str:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "UNKNOWN"

print("✅ All deps installed")
print(f"  munch={get_package_version('munch')}")
print(f"  noisereduce={get_package_version('noisereduce')}")
print(f"  phonemizer={get_package_version('phonemizer')}")
print(f"  espeakng-loader={get_package_version('espeakng-loader')}")
print(f"  click={get_package_version('click')}")

In [ ]:
# Cài Monotonic Alignment Search cho StyleTTS2
# Dùng %pip để cài đúng vào Python environment của kernel Kaggle hiện tại.
!apt-get update -qq
!apt-get install -y -qq build-essential python3-dev

%pip uninstall -y monotonic-align
%pip install -q --upgrade pip setuptools wheel cython
%pip install --no-cache-dir --no-build-isolation \
    "git+https://github.com/resemble-ai/monotonic_align.git"

In [ ]:
# Kiểm tra monotonic
import sys
import monotonic_align

from monotonic_align import maximum_path, mask_from_lens
from monotonic_align.core import maximum_path_c

print("✅ monotonic_align installed")
print("Python executable :", sys.executable)
print("Package location  :", monotonic_align.__file__)
print("maximum_path      :", maximum_path)
print("mask_from_lens     :", mask_from_lens)
print("maximum_path_c     :", maximum_path_c)

## Bước 5 — Tải pretrained checkpoint Vietnamese

Tải `model.pth` (~570 MB) và `config.yaml` từ HuggingFace `dangtr0408/StyleTTS2-lite-vi/Models/`.

**LƯU Ý**: Repo gốc yêu cầu đặt pretrained ở `Models/Finetune/base_model.pth` + config tương ứng ở `Configs/`.

In [ ]:
import os, urllib.request

MODELS_DIR = f"{REPO_DIR}/Models/Finetune"
CONFIGS_DIR = f"{REPO_DIR}/Configs"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CONFIGS_DIR, exist_ok=True)

# 1. Download pretrained model
MODEL_URL = "https://huggingface.co/dangtr0408/StyleTTS2-lite-vi/resolve/main/Models/base_model_120k_vi.pth"
MODEL_PATH = f"{MODELS_DIR}/base_model_120k_vi.pth"

if not os.path.exists(MODEL_PATH) or os.path.getsize(MODEL_PATH) < 500_000_000:
    print(f"Downloading pretrained model (~570 MB)...")
    !wget -q --show-progress "{MODEL_URL}" -O "{MODEL_PATH}"
else:
    print(f"Pretrained đã có: {MODEL_PATH}")

# 2. Download config gốc (làm template)
CONFIG_URL = "https://huggingface.co/dangtr0408/StyleTTS2-lite-vi/resolve/main/Models/config.yaml"
ORIGINAL_CONFIG = f"{CONFIGS_DIR}/config_lite_vi_original.yaml"

if not os.path.exists(ORIGINAL_CONFIG):
    print(f"Downloading config gốc...")
    !wget -q "{CONFIG_URL}" -O "{ORIGINAL_CONFIG}"
else:
    print(f"Config gốc đã có: {ORIGINAL_CONFIG}")

# Verify
import os
print(f"\nModel  : {MODEL_PATH}  ({os.path.getsize(MODEL_PATH)/1e6:.1f} MB)")
print(f"Config : {ORIGINAL_CONFIG}  ({os.path.getsize(ORIGINAL_CONFIG)} bytes)")


In [ ]:
# Kiểm tra checkpoint
from pathlib import Path
import torch

PRETRAINED_PATH = Path(
    "/kaggle/working/StyleTTS2-lite/Models/Finetune/base_model_120k_vi.pth"
)

assert PRETRAINED_PATH.exists(), f"Không tìm thấy: {PRETRAINED_PATH}"

size_bytes = PRETRAINED_PATH.stat().st_size

print(f"Checkpoint size: {size_bytes:,} bytes")
print(f"Checkpoint size: {size_bytes / (1024 ** 3):.3f} GiB")

assert size_bytes > 1_000_000_000, (
    "Checkpoint vẫn quá nhỏ. Có thể file chưa tải xong hoặc bị lỗi."
)

state = torch.load(
    PRETRAINED_PATH,
    map_location="cpu",
    weights_only=False,
)

print("✅ Checkpoint load thành công")
print("Type:", type(state))

if isinstance(state, dict):
    print("Keys:", list(state.keys())[:20])

## Bước 6 — ⚙️ Session configuration (BẠN PHẢI EDIT)

Thay đổi 3 biến dưới đây tùy theo phiên hiện tại:

### Phiên 1 (lần đầu fine-tune):
```python
IS_RESUME = False
PREVIOUS_CKPT_DATASET = ""
PREVIOUS_CKPT_FILENAME = ""
```

### Phiên 2+ (resume từ checkpoint phiên trước):
1. Phiên trước, sau khi Save Version → download `current_model.pth` từ Output
2. Tạo Kaggle Dataset mới (vd: `ngan-checkpoint-v1`) chứa file `current_model.pth`
3. Attach dataset đó vào notebook này (+Add Data)
4. Set:
   ```python
   IS_RESUME = True
   PREVIOUS_CKPT_DATASET = "vieanh88/ngan-checkpoint-v1"   # slug dataset
   PREVIOUS_CKPT_FILENAME = "current_model.pth"            # tên file trong dataset
   ```

In [ ]:
# ===== EDIT 3 BIẾN NÀY =====
IS_RESUME = False
PREVIOUS_CKPT_DATASET = ""
PREVIOUS_CKPT_FILENAME = ""

# Hyperparameters (đã chọn phù hợp T4 x2 + 2000 samples)
BATCH_SIZE = 2          # Tổng — DataParallel chia 1/GPU. Tăng 4-6 nếu OK.
MAX_LEN = 310           # Mel frames (~3.875s). Giảm xuống 250 nếu OOM.
EPOCHS = 20             # Tổng cộng (qua tất cả phiên). Sẽ resume start_epoch.
SAVE_FREQ = 1           # Save best mỗi N epoch
LOG_INTERVAL = 50       # Log mỗi N iterations
LEARNING_RATE = 1e-4    # Base lr cho text_encoder, text_aligner, predictor
FT_LEARNING_RATE = 1e-5 # Fine-tune lr cho decoder, style_encoder

# ===== DERIVED VALUES (KHÔNG SỬA) =====
if IS_RESUME:
    if not PREVIOUS_CKPT_DATASET or not PREVIOUS_CKPT_FILENAME:
        raise ValueError("IS_RESUME=True nhưng chưa điền PREVIOUS_CKPT_DATASET/FILENAME")
    # Path checkpoint trong /kaggle/input
    PRETRAINED_PATH = f"/kaggle/input/{PREVIOUS_CKPT_DATASET.split('/')[-1]}/{PREVIOUS_CKPT_FILENAME}"
    LOAD_ONLY_PARAMS = False  # Load CẢ optimizer + epoch counter
    print(f"📂 RESUME mode")
    print(f"   Pretrained: {PRETRAINED_PATH}")
    if not os.path.exists(PRETRAINED_PATH):
        raise RuntimeError(
            f"❌ Không thấy {PRETRAINED_PATH}\n"
            f"   Đã attach dataset '{PREVIOUS_CKPT_DATASET}' chưa?"
        )
else:
    PRETRAINED_PATH = MODEL_PATH  # base_model.pth từ Step 5
    LOAD_ONLY_PARAMS = True   # Fresh start, không load optimizer cũ của viVoice
    print(f"📂 FIRST session — train từ pretrained Vietnamese")
    print(f"   Pretrained: {PRETRAINED_PATH}")

print(f"\nHyperparameters:")
print(f"  batch_size  : {BATCH_SIZE}")
print(f"  max_len     : {MAX_LEN}")
print(f"  epochs      : {EPOCHS}")
print(f"  lr / ft_lr  : {LEARNING_RATE} / {FT_LEARNING_RATE}")


## Bước 7 — Generate config YAML

Đọc config gốc của lite-vi làm template → override với hyperparameters + path Kaggle → ghi ra file mới.

In [ ]:
import yaml, shutil
from pathlib import Path

# Đọc config gốc
with open(ORIGINAL_CONFIG, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# ====== Override các fields cho Kaggle ======
cfg["log_dir"] = "./Models/Finetune"   # Relative tới repo dir
cfg["save_freq"] = SAVE_FREQ
cfg["log_interval"] = LOG_INTERVAL
cfg["device"] = "cuda"
cfg["epochs"] = EPOCHS
cfg["batch_size"] = BATCH_SIZE
cfg["max_len"] = MAX_LEN
cfg["pretrained_model"] = PRETRAINED_PATH
cfg["load_only_params"] = LOAD_ONLY_PARAMS
cfg["debug"] = True   # In warning UNKNOWN IPA CHARACTERS từ TextCleaner

# Data paths — TUYỆT ĐỐI vào /kaggle/input/
cfg["data_params"] = {
    "train_data": TRAIN_FILE,
    "val_data": VAL_FILE,
    "root_path": DATA_ROOT + "/",   # ghép với 'wavs/ngan_xxx.wav' trong filelist
}

# Training strats — KHÔNG freeze, KHÔNG ignore (default)
# Style encoder vẫn train nhưng đã có ft_lr thấp -> giữ multi-speaker ability đủ ổn
cfg["training_strats"] = {
    "freeze_modules": [""],
    "ignore_modules": [""],
}

# Optimizer params
cfg["optimizer_params"] = {
    "lr": LEARNING_RATE,
    "ft_lr": FT_LEARNING_RATE,
}

# ====== Write file ======
CONFIG_FILE = f"{CONFIGS_DIR}/config_ngan_kaggle.yml"
with open(CONFIG_FILE, "w", encoding="utf-8") as f:
    yaml.dump(cfg, f, allow_unicode=True, sort_keys=False, default_flow_style=False)

print(f"✅ Config written: {CONFIG_FILE}")
print(f"\nNội dung config (10 dòng đầu):")
with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 30: break
        print(f"  {line.rstrip()}")
print("  ...")


## Bước 8 — (Optional) Dry-run sanity check

**Tốn ~2 phút** nhưng phát hiện sớm vấn đề (path sai, vocab mismatch, OOM) TRƯỚC KHI tốn 8 giờ training.

Chạy cell này 1 lần ở phiên đầu tiên. Nếu OK → có thể skip ở các phiên sau.

In [ ]:
from pathlib import Path

PRETRAINED_PATH = Path(
    "/kaggle/working/StyleTTS2-lite/Models/Finetune/base_model_120k_vi.pth"
)

print("Path   :", PRETRAINED_PATH)
print("Exists :", PRETRAINED_PATH.exists())

if PRETRAINED_PATH.exists():
    size_bytes = PRETRAINED_PATH.stat().st_size

    print(f"Size   : {size_bytes:,} bytes")
    print(f"Size   : {size_bytes / (1024 ** 3):.3f} GiB")

    with PRETRAINED_PATH.open("rb") as f:
        first_bytes = f.read(200)

    print("First bytes:", repr(first_bytes))

In [ ]:
import yaml
from pathlib import Path

CONFIG_FILE = Path("/kaggle/working/StyleTTS2-lite/Configs/config_ngan_kaggle.yml")

with CONFIG_FILE.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

sym = config["symbol"]

symbols = (
    list(sym["pad"])
    + list(sym["punctuation"])
    + list(sym["letters"])
    + list(sym["letters_ipa"])
    + list(sym["extend"])
)

symbol_dict = {ch: idx for idx, ch in enumerate(symbols)}
n_token = len(symbol_dict) + 1

print("Config file       :", CONFIG_FILE)
print("Raw symbol count  :", len(symbols))
print("Unique symbols    :", len(symbol_dict))
print("Calculated n_token:", n_token)

In [ ]:
# Sửa .squeeze() thành .squeeze(-1) để chỉ xóa chiều class cuối cùng, không bao giờ xóa chiều batch.
# Kết quả cần thấy: return torch.abs(classifier_out.squeeze(-1)), GAN_feature, poolblock_out
from pathlib import Path
import re

REPO_DIR = Path("/kaggle/working/StyleTTS2-lite")
JDC_FILE = REPO_DIR / "Modules" / "JDC" / "model.py"

assert JDC_FILE.exists(), f"Không tìm thấy file: {JDC_FILE}"

source = JDC_FILE.read_text(encoding="utf-8")

pattern = (
    r"return\s+torch\.abs\(classifier_out\.squeeze\(\)\),"
    r"\s*GAN_feature,\s*poolblock_out"
)

replacement = (
    "return torch.abs(classifier_out.squeeze(-1)), "
    "GAN_feature, poolblock_out"
)

patched_source, replacement_count = re.subn(
    pattern,
    replacement,
    source,
)

if replacement_count > 0:
    JDC_FILE.write_text(patched_source, encoding="utf-8")
    print(f"✅ Đã patch {replacement_count} dòng trong: {JDC_FILE}")
elif replacement in source:
    print("✅ File JDCNet đã được patch trước đó.")
else:
    raise RuntimeError(
        "Không tìm thấy dòng classifier_out.squeeze() để patch.\n"
        "Hãy mở Modules/JDC/model.py và kiểm tra hàm JDCNet.forward()."
    )

print("\nCác dòng chứa classifier_out.squeeze:")
for line_number, line in enumerate(
    JDC_FILE.read_text(encoding="utf-8").splitlines(),
    start=1,
):
    if "classifier_out.squeeze" in line:
        print(f"{line_number}: {line}")

In [ ]:
# Test 1: Load model + checkpoint
%cd {REPO_DIR}
import sys
sys.path.insert(0, REPO_DIR)

import yaml, torch
from munch import Munch
from models import build_model
from utils import recursive_munch

with open(CONFIG_FILE) as f:
    test_cfg = yaml.safe_load(f)

symbols = (
    list(test_cfg['symbol']['pad']) +
    list(test_cfg['symbol']['punctuation']) +
    list(test_cfg['symbol']['letters']) +
    list(test_cfg['symbol']['letters_ipa']) +
    list(test_cfg['symbol']['extend'])
)
n_token = len(symbols) + 1
print(f"Test 1: Build model with n_token={n_token}")

model_params = recursive_munch(test_cfg['model_params'])
model_params['n_token'] = n_token
model = build_model(model_params)
print(f"  ✅ Built {len(model)} modules: {list(model.keys())}")

# Test 2: Load checkpoint state
print(f"\nTest 2: Load checkpoint {PRETRAINED_PATH}")
state = torch.load(PRETRAINED_PATH, map_location='cpu', weights_only=False)
print(f"  ✅ Checkpoint keys: {list(state.keys())}")
if 'epoch' in state:
    print(f"  Previous epoch: {state['epoch']}")
if 'iters' in state:
    print(f"  Previous iters: {state['iters']}")


In [ ]:
# Test 3: Build dataloader + iterate 1 batch
from meldataset import build_dataloader
from utils import get_data_path_list

train_list, val_list = get_data_path_list(TRAIN_FILE, VAL_FILE)
print(f"Test 3: Loaded {len(train_list)} train, {len(val_list)} val records")

symbol_dict = {s: i for i, s in enumerate(symbols)}

test_dl = build_dataloader(
    train_list[:10],         # 10 sample để test nhanh
    test_cfg['data_params']['root_path'],
    symbol_dict,
    batch_size=2,
    num_workers=1,
    dataset_config={"debug": True},
    device='cuda'
)

# Iterate 1 batch
for batch in test_dl:
    waves, texts, input_lengths, mels, mel_input_length = batch
    print(f"  ✅ Batch shapes:")
    print(f"     waves[0] : {len(waves[0])} samples")
    print(f"     texts    : {texts.shape}")
    print(f"     mels     : {mels.shape}")
    print(f"     mel_len  : {mel_input_length}")
    break
print("\n✅ Dry-run PASS — sẵn sàng train!")


In [ ]:
# Kiểm tra patch với 2 GPU
# Chạy cell này để test bằng một tiến trình Python mới, tránh việc notebook đang cache module cũ.
# Kết quả mong đợi trên Kaggle 2 GPU:
#  GPU count : 2
#  Input     : (2, 1, 80, 310)
#  F0 output : (2, 310)
#  ✅ JDCNet DataParallel shape test PASS
import subprocess
import sys
import textwrap
from pathlib import Path

REPO_DIR = Path("/kaggle/working/StyleTTS2-lite")

test_code = textwrap.dedent(
    """
    import torch
    from Modules.JDC.model import JDCNet

    assert torch.cuda.is_available(), "CUDA không khả dụng"

    num_gpus = torch.cuda.device_count()
    batch_size = max(2, num_gpus)
    num_frames = 310

    model = JDCNet(num_class=1, seq_len=192).cuda()

    if num_gpus > 1:
        model = torch.nn.DataParallel(model)

    x = torch.randn(
        batch_size,
        1,
        80,
        num_frames,
        device="cuda",
    )

    with torch.no_grad():
        f0, _, _ = model(x)

    print("GPU count :", num_gpus)
    print("Input     :", tuple(x.shape))
    print("F0 output :", tuple(f0.shape))

    expected_shape = (batch_size, num_frames)

    assert tuple(f0.shape) == expected_shape, (
        f"Shape sai: nhận {tuple(f0.shape)}, cần {expected_shape}"
    )

    print("✅ JDCNet DataParallel shape test PASS")
    """
)

subprocess.run(
    [sys.executable, "-c", test_code],
    cwd=str(REPO_DIR),
    check=True,
)

## Bước 9 — 🚀 START TRAINING (cell chính, sẽ chạy 7-8 giờ)

> 🔔 **NGAY KHI cell này bắt đầu chạy**: bật timer điện thoại **7 giờ**.  
> Sau 7h → quay lại notebook → click **"Save Version" → "Quick Save"** → đợi save xong → mới đóng tab.

**Cách monitor**:
- Log mel_loss / dur_loss in mỗi 50 iterations
- TensorBoard log tại `Models/Finetune/tensorboard/` (cell tiếp theo launch)
- File checkpoint cập nhật mỗi 1000 iter (`current_model.pth`) + mỗi epoch tốt hơn (`epoch_XXXXX.pth`)

**Stop sớm**: nếu val loss tăng liên tục 3-5 epochs → có thể Ctrl+M I (interrupt kernel) và save.

In [ ]:
# Reset CWD, free CUDA cache, start training
%cd {REPO_DIR}
import gc, torch
gc.collect()
torch.cuda.empty_cache()

# Print free VRAM trước khi start
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {free/1e9:.1f}/{total/1e9:.1f} GB free")

print(f"\n{'='*60}")
print(f"BẮT ĐẦU TRAINING")
print(f"{'='*60}")
print(f"Config       : {CONFIG_FILE}")
print(f"Pretrained   : {PRETRAINED_PATH}")
print(f"Epochs target: {EPOCHS}")
print(f"\nLog_dir output: {REPO_DIR}/Models/Finetune/")
print(f"  - current_model.pth  (cập nhật mỗi 1000 iters — DÙNG ĐỂ RESUME)")
print(f"  - epoch_XXXXX.pth    (best mỗi epoch val loss tốt hơn)")
print(f"  - tensorboard/       (TensorBoard log)")
print(f"\n⏰ NHẮC: bật timer 7 giờ NGAY BÂY GIỜ!\n")

# Run training (relative path để train.py đọc đúng log_dir)
!python train.py -p Configs/config_ngan_kaggle.yml


## Bước 10 — (Optional) Launch TensorBoard

Chạy cell này TRONG TAB BROWSER KHÁC, hoặc sau khi training xong. KHÔNG launch cùng cell train (sẽ block UI).

In [ ]:
# Load TensorBoard extension (Kaggle có sẵn)
%load_ext tensorboard

# Launch — sẽ hiển thị inline trong cell output
%tensorboard --logdir {REPO_DIR}/Models/Finetune/tensorboard --port 6006


## Bước 11 — Trước khi Save Version

Sau khi training dừng (kết thúc epochs hoặc bạn interrupt), chạy cell dưới để:
- List checkpoint đã save
- Verify file `current_model.pth` exists (dùng để resume phiên sau)
- Tổng kết kích thước

In [ ]:
import os
from pathlib import Path

CKPT_DIR = f"{REPO_DIR}/Models/Finetune"
print(f"Checkpoints saved tại: {CKPT_DIR}\n")

# List checkpoints
files = sorted(Path(CKPT_DIR).glob("*.pth"))
if not files:
    print("⚠️  KHÔNG có file .pth nào — training có vấn đề?")
else:
    print(f"Tìm thấy {len(files)} checkpoint(s):")
    total_size = 0
    for f in files:
        size_mb = f.stat().st_size / 1e6
        total_size += size_mb
        print(f"  {f.name:30s}  {size_mb:7.1f} MB")
    print(f"  {'TOTAL':30s}  {total_size:7.1f} MB")

# Verify current_model.pth (CRITICAL cho resume)
current = Path(CKPT_DIR) / "current_model.pth"
if current.exists():
    print(f"\n✅ current_model.pth EXISTS — dùng để resume phiên sau")
else:
    print(f"\n⚠️  current_model.pth KHÔNG có — chưa đến iter 1000?")
    print(f"   Resume sẽ dùng epoch_XXXXX.pth mới nhất")

# Kiểm tra disk usage tổng của /kaggle/working
print(f"\nDisk usage /kaggle/working:")
!du -sh /kaggle/working/


## Bước 12 — 💾 Save Version & Download checkpoint

### A. Save Version trên Kaggle UI

1. Click nút **"Save Version"** ở góc trên phải notebook
2. Popup chọn:
   - **"Quick Save"** (RECOMMEND nếu vẫn đang trong session): chỉ save state hiện tại, KHÔNG re-run notebook
   - "Save & Run All": re-run từ đầu → KHÔNG dùng vì sẽ train lại từ đầu, mất checkpoint
3. Đặt Version name, vd: `phien-1-epoch-15-loss-X.X`
4. Click "Save"
5. Đợi ~2-5 phút cho Kaggle commit

### B. Download checkpoint về máy local

Sau khi Save Version xong:
1. Vào Version vừa tạo → tab **"Output"**
2. Navigate `working/StyleTTS2-lite/Models/Finetune/`
3. Click file `current_model.pth` → "Download"
4. (Optional) Cũng download `epoch_XXXXX.pth` mới nhất (best model)

### C. Tạo Kaggle Dataset cho phiên sau

1. Click **"+ New Dataset"**
2. Drag file `current_model.pth` đã download
3. Title: `ngan-checkpoint-v1` (hoặc v2, v3 cho các phiên sau)
4. Visibility: Private
5. Create

### D. Phiên 2+ — Resume

1. Tạo Kaggle Notebook mới (hoặc fork notebook này)
2. Attach 2 datasets: `ngan-data-lite-vi` + `ngan-checkpoint-v1`
3. Trong Cell **Step 6**, đổi:
   ```python
   IS_RESUME = True
   PREVIOUS_CKPT_DATASET = "vieanh88/ngan-checkpoint-v1"
   PREVIOUS_CKPT_FILENAME = "current_model.pth"
   ```
4. Run All → training tiếp tục từ epoch dở dang

---

## Tổng kết hiệu suất dự kiến

| Phiên | Epochs | Thời gian | Best val mel loss kỳ vọng |
|-------|--------|-----------|---------------------------|
| 1 | 0 → ~15 | 7-8h | giảm nhanh, ~0.5 |
| 2 | 15 → ~30 | 7-8h | converge, ~0.3 |
| 3 | 30 → ~45 | 7-8h | refine, ~0.25 |

Nếu val loss giảm chững → có thể dừng sớm ở phiên 2 (epoch ~25-30) và đi inference test.

---

## Sau khi train xong

→ Đi tiếp **D1** (download_female_ref.py — lấy reference giọng nữ) → **D2** (nlp_generator.py — Gemini API) → **D3** (tts_generator.py — sinh audio horror).